Eddy Guo

eyguo-code

8913122458

1.c.i

From research on the internet, some features usually used in time series classification include: 
- Minimum and Maximum
- Mean 
- Median
- Variance and Standard Deviation
- Skewness
- Kurtosis
- Autocorrelation
- Partial Autocorrelation
- Zero-Crossing Rate
- Root Mean Square
- Hjorth Parameters like Comprises Activity, Mobility, Complexity 

1.c.ii

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np

data_dir = Path(
    "../Data/activity+recognition+system+based+on+multisensor+data+fusion+arem"
)

csv_files = sorted(data_dir.rglob("*.csv"))

# print(f"Found {len(csv_files)} CSV files")

statistics = [
    "min", "max", "mean", "median",
    "std", "1st_quart", "3rd_quart"
]

feature_columns = [
    f"{stat}{i}"
    for i in range(1, 7)
    for stat in statistics
]

feature_rows = []

for file_path in csv_files:

    df = pd.read_csv(
        file_path,
        skiprows=5,
        header=None,
        usecols=range(1, 7),
        sep=r",\s*|\s+",
        engine="python"
    )

    row = []

    for col in df.columns:
        series = pd.to_numeric(df[col], errors="coerce")

        row.extend([
            series.min(),
            series.max(),
            series.mean(),
            series.median(),
            series.std(),
            series.quantile(0.25),
            series.quantile(0.75)
        ])

    feature_rows.append(row)

features_df = pd.DataFrame(
    feature_rows,
    columns=feature_columns
)

features_df.insert(
    0, "Instance", range(1, len(features_df) + 1)
)

# print("Final dataset shape:", features_df.shape)
display(features_df)


Final dataset shape: (88, 43)


,Instance,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,1,37.25,45.00,40.624792,40.500,1.476967,39.25,42.0000,0.0,1.30,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.430,0.582915,0.00,1.3000
1,2,38.00,45.67,42.812812,42.500,1.435550,42.00,43.6700,0.0,1.22,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.430,0.601010,0.00,1.3000
2,3,35.00,47.40,43.954500,44.330,1.558835,43.00,45.0000,0.0,1.70,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.430,0.513506,0.00,0.9400
3,4,33.00,47.75,42.179812,43.500,3.670666,39.15,45.0000,0.0,3.00,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.500,0.524317,0.00,1.0000
4,5,33.00,45.75,41.678063,41.750,2.243490,41.33,42.7500,0.0,2.83,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.430,0.389164,0.00,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,84,20.75,46.25,34.763333,35.290,4.742208,31.67,38.2500,0.0,12.68,...,3.174681,14.2500,18.33,0.0,9.39,3.288271,3.270,1.647528,2.05,4.3050
84,85,21.50,51.00,34.935812,35.500,4.645944,32.00,38.0625,0.0,12.21,...,3.192058,14.2375,18.25,0.0,10.21,3.280021,3.015,1.700918,2.12,4.5000
85,86,18.33,47.67,34.333042,34.750,4.948770,31.25,38.0000,0.0,12.48,...,3.000493,13.7500,18.00,0.0,8.01,3.261583,2.980,1.617290,2.05,4.3200
86,87,18.33,45.75,34.599875,35.125,4.731790,31.50,38.0000,0.0,15.37,...,2.905688,14.0000,18.25,0.0,8.86,3.289542,3.015,1.680170,2.12,4.2600


1.c.iii.

In [ ]:
X = features_df.drop(columns=["Instance"])

n_bootstrap = 10000
confidence_level = 0.90
rng = np.random.default_rng(42)

results = []

for feature in X.columns:
    data = X[feature].dropna().to_numpy()
    n = len(data)

    estimated_std = np.std(data, ddof=1)

    bootstrap_stds = []

    for _ in range(n_bootstrap):
        sample = rng.choice(data, size=n, replace=True)
        bootstrap_stds.append(np.std(sample, ddof=1))

    lower_bound = np.percentile(bootstrap_stds, 5)
    upper_bound = np.percentile(bootstrap_stds, 95)

    results.append({
        "Feature": feature,
        "Estimated Std": estimated_std,
        "90% CI Lower": lower_bound,
        "90% CI Upper": upper_bound
    })

std_ci_df = pd.DataFrame(results)

display(std_ci_df.round(4))


,Feature,Estimated Std,90% CI Lower,90% CI Upper
0,min1,9.5700,8.2591,10.7762
1,max1,4.3944,3.3417,5.2936
2,mean1,5.3357,4.6993,5.8818
3,median1,5.4401,4.7910,5.9913
4,std1,1.7722,1.5726,1.9492
5,1st_quart1,6.1536,5.5814,6.6466
6,3rd_quart1,5.1389,4.3177,5.8483
7,min2,0.0000,0.0000,0.0000
8,max2,5.0627,4.6241,5.4018
9,mean2,1.5742,1.3929,1.7071


1.c.iv.

For this dataset and the information form the previous cells, I could only choose max, mean, and std dev.

2.

3.7.4.

(a) The cubic model would have at most the same RSS as the linear one. This is because there are more polynomial terms in the cubic model that allows it to possibly fit some noise more than the linear model. 

(b) For the test RSS, the linear model would have lower RSS. This is because the true relationship is linear and thus the cubic model might overfit.

(c) We would still expect the cubic model to have lower training RSS due to its flexibility. 

(d) We would need more information to judge. 